# Day 023 — Exercise 5: ai_extract_from_page

**What you'll build:** `ai_extract_from_page(html_content, question, model)` — strips HTML tags to plain text, then asks the LLM a question about the content.

**Why it matters:** Scraped HTML is messy. Instead of writing fragile CSS selectors for every page layout, you can strip tags and let the LLM extract or answer questions in plain English — adaptive to any page structure.

In [ ]:
import ollama
from bs4 import BeautifulSoup

## Your Implementation

In [ ]:
def ai_extract_from_page(html_content: str, question: str, model: str = "llama3.2") -> str:
    """
    Strip HTML tags and ask the LLM a question about the page content.

    Args:
        html_content: Raw HTML string.
        question:     Plain-English question about the page.
        model:        Ollama model name.

    Returns:
        LLM answer as a string. Never raises.
    """
    # TODO: soup = BeautifulSoup(html_content, 'html.parser')
    # TODO: for tag in soup(['script', 'style']): tag.decompose()
    # TODO: text = soup.get_text(separator="\n", strip=True)
    # TODO: call ollama.chat with system='web page analyst' and
    #       user content = page text capped at 3000 chars + question
    # TODO: return response['message']['content']
    pass

## Check Your Work

In [ ]:
HTML = '<!DOCTYPE html>\n<html>\n<head><title>Book Reviews</title></head>\n<body>\n  <h1>Top Books</h1>\n  <p class="intro">Curated reading list.</p>\n  <ul>\n    <li><a href="/books/python">Learn Python</a></li>\n    <li><a href="/books/ml">Machine Learning</a></li>\n    <li><a href="/books/data">Data Science</a></li>\n    <li><a>No Link</a></li>\n  </ul>\n  <h2>Fiction</h2>\n  <h2>Non-Fiction</h2>\n  <p class="highlight">Great content here.</p>\n  <p class="highlight">More highlights.</p>\n</body>\n</html>'


def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined
    try:
        assert 'ai_extract_from_page' in globals()
        passed += 1; print('\u2705 Check 1: ai_extract_from_page defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    result = None

    # Check 2: returns a string (1 LLM call)
    try:
        result = ai_extract_from_page(HTML, 'What is the main topic of this page?')
        assert isinstance(result, str), \
            f'expected str, got {type(result)}'
        passed += 1; print('\u2705 Check 2: returns a string')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: result is non-empty
    try:
        assert result is not None, 'result is None (Check 2 failed)'
        assert len(result) > 0, 'result is empty string'
        passed += 1; print('\u2705 Check 3: result is non-empty')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: result is a meaningful response
    try:
        assert result is not None, 'result is None'
        assert len(result) > 10, \
            f'result too short ({len(result)} chars): {result!r}'
        passed += 1; print(f'\u2705 Check 4: result has {len(result)} chars')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: works on minimal HTML — no crash (1 LLM call)
    try:
        empty_result = ai_extract_from_page('<html><body></body></html>', 'Any content?')
        assert isinstance(empty_result, str), \
            f'expected str for minimal HTML, got {type(empty_result)}'
        passed += 1; print('\u2705 Check 5: works on minimal HTML without crashing')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def ai_extract_from_page(html_content: str, question: str, model: str = "llama3.2") -> str:
    soup = BeautifulSoup(html_content, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are a web page analyst. Answer questions about the page content concisely.",
            },
            {
                "role": "user",
                "content": f"Page content:\n{text[:3000]}\n\nQuestion: {question}",
            },
        ],
    )
    return response["message"]["content"]
```

</details>